# 04 Discount Analysis

This notebook answers the core pricing questions about discount prevalence, discount depth, and discount patterns by category and brand.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# File paths
orders_path = '/mnt/data/orders_cl.csv'
products_path = '/mnt/data/products_cl(1).csv'
orderlines_path = '/mnt/data/orderlines_qu.csv'
brands_path = '/mnt/data/brands(1).csv'

# Load datasets
orders_cl = pd.read_csv(orders_path)
products_cl = pd.read_csv(products_path)
orderlines_qu = pd.read_csv(orderlines_path)
brands_df = pd.read_csv(brands_path)

print('orders_cl:', orders_cl.shape)
print('products_cl:', products_cl.shape)
print('orderlines_qu:', orderlines_qu.shape)
print('brands_df:', brands_df.shape)

orders_clean = orders_cl.drop_duplicates().copy()
products_clean = products_cl.drop_duplicates(subset='sku').copy()
orderlines_clean = orderlines_qu.drop_duplicates().copy()
brands_clean = brands_df.drop_duplicates().copy()

orders_clean['created_date'] = pd.to_datetime(orders_clean['created_date'], errors='coerce')
orderlines_clean['date'] = pd.to_datetime(orderlines_clean['date'], errors='coerce')
orders_clean['total_paid'] = pd.to_numeric(orders_clean['total_paid'], errors='coerce')
products_clean['price'] = pd.to_numeric(products_clean['price'], errors='coerce')
orderlines_clean['unit_price'] = pd.to_numeric(orderlines_clean['unit_price'], errors='coerce')
orderlines_clean['product_quantity'] = pd.to_numeric(orderlines_clean['product_quantity'], errors='coerce')

orders_clean = orders_clean.dropna(subset=['order_id', 'created_date', 'total_paid', 'state'])
products_clean = products_clean.dropna(subset=['sku', 'name', 'price'])
orderlines_clean = orderlines_clean.dropna(subset=['id_order', 'sku', 'unit_price', 'product_quantity', 'date'])
products_clean = products_clean[products_clean['price'] > 0]
orderlines_clean = orderlines_clean[(orderlines_clean['unit_price'] > 0) & (orderlines_clean['product_quantity'] > 0)]

products_clean['brand_code'] = products_clean['sku'].str[:3]
products_clean = products_clean.merge(brands_clean, left_on='brand_code', right_on='short', how='left')
products_clean = products_clean.rename(columns={'long': 'brand_name'})
products_clean['text'] = (products_clean['name'].fillna('') + ' ' + products_clean['desc'].fillna('')).str.lower()

def categorize_product(text):
    if ('iphone' in text) and ('case' not in text):
        return 'Smartphone'
    elif 'ipad' in text:
        return 'Tablet'
    elif ('macbook' in text) or ('imac' in text) or ('mac mini' in text) or ('mac pro' in text):
        return 'Computer'
    elif ('ssd' in text) or ('hard drive' in text) or ('external drive' in text) or ('nas' in text):
        return 'Storage'
    elif ('case' in text) or ('cable' in text) or ('adapter' in text) or ('charger' in text):
        return 'Peripheral'
    elif ('keyboard' in text) or ('mouse' in text) or ('headphone' in text) or ('speaker' in text):
        return 'Accessory'
    else:
        return 'Other'

products_clean['category'] = products_clean['text'].apply(categorize_product)

sales_df = orderlines_clean.merge(
    products_clean[['sku', 'price', 'brand_name', 'category']],
    on='sku',
    how='left'
).merge(
    orders_clean[['order_id', 'created_date', 'state']],
    left_on='id_order',
    right_on='order_id',
    how='left'
)

sales_df['revenue'] = sales_df['unit_price'] * sales_df['product_quantity']
sales_df['discount'] = sales_df['price'] - sales_df['unit_price']
sales_df['discount_pct'] = (sales_df['discount'] / sales_df['price']) * 100
sales_df['is_discounted'] = sales_df['unit_price'] < sales_df['price']


## 1. How many products are being discounted?

In [ ]:
discounted_products = sales_df.loc[sales_df['is_discounted'], 'sku'].nunique()
total_products_sold = sales_df['sku'].nunique()
discounted_share = round((discounted_products / total_products_sold) * 100, 2)

print(f'{discounted_products} out of {total_products_sold} sold SKUs were discounted at least once ({discounted_share}%).')

In [ ]:
counts = pd.Series({
    'Not discounted': sales_df.loc[~sales_df['is_discounted'], 'sku'].nunique(),
    'Discounted': sales_df.loc[sales_df['is_discounted'], 'sku'].nunique()
})

plt.figure(figsize=(6,4))
counts.plot(kind='bar')
plt.title('Discounted vs non-discounted sold SKUs')
plt.ylabel('Unique SKUs')
plt.xticks(rotation=0)
plt.show()

## 2. How big are the offered discounts?

In [ ]:
discount_summary = sales_df.loc[sales_df['is_discounted'], 'discount_pct'].describe().round(2)
display(discount_summary)

## 3. Discount depth by category

In [ ]:
discount_by_category = (
    sales_df.groupby('category')['discount_pct']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
display(discount_by_category)

plt.figure(figsize=(8,4))
discount_by_category.plot(kind='bar')
plt.title('Average discount % by category')
plt.ylabel('Average discount (%)')
plt.xticks(rotation=45)
plt.show()

## 4. Share of discounted sold SKUs by category

In [ ]:
discount_share_by_category = (
    sales_df.groupby('category')['is_discounted']
    .mean()
    .sort_values(ascending=False) * 100
).round(2)
display(discount_share_by_category)

plt.figure(figsize=(8,4))
discount_share_by_category.plot(kind='bar')
plt.title('Discounted transaction share by category')
plt.ylabel('Discounted transactions (%)')
plt.xticks(rotation=45)
plt.show()

## 5. Top brands by revenue

In [ ]:
top10_brands = (
    sales_df.groupby('brand_name')['revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .round(2)
)
display(top10_brands)

plt.figure(figsize=(9,4))
top10_brands.plot(kind='bar')
plt.title('Top 10 brands by revenue')
plt.ylabel('Revenue (€)')
plt.xticks(rotation=45, ha='right')
plt.show()

## 6. Business interpretation

Fill this markdown cell with your conclusions for the board.